# 00 · Environment and toolkit check

Checks the Kaggle environment, Hugging Face access, and installs the four Lexsi toolkits so we can record their real APIs before writing code against them (docs/03 §5, ADR-0003).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 or P100 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) |
| **Reads** | nothing |
| **Writes** (`/kaggle/working/`) | `toolkits/toolkit_apis.json` |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

In [1]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

{'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'cuda_available': True,
 'on_kaggle': True,
 'smoke': False,
 'peft': '0.21.2',
 'gpus': ['Tesla T4', 'Tesla T4'],
 'capability': [7, 5],
 'vram_gb': 15.6}

In [2]:
sp.hf_login()
tok = sp.load_tokenizer()           # proves internet + Hub access
print("tokenizer ok, vocab", len(tok))

No HF_TOKEN found: gated models/datasets (e.g. Llama Guard) will not load.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

tokenizer ok, vocab 151665


## Lexsi toolkits

Install commands and import names are copied from each toolkit's README on github.com/Lexsi-Labs
(checked 2026-11-06). The cell below records where pip found each package, so check that `home` points to
Lexsi-Labs: typo-squatted packages exist. SafeTune needs Python ≥ 3.12 and fails cleanly on older Pythons.

Installing may change torch/transformers versions: compare the `env` block before and after, and restart
the kernel before using a toolkit in a later cell.

In [3]:
import platform
print("python", platform.python_version())
TOOLKITS = {
    "CuratorKIT": {"install": "curatorkit[all]", "import": "curatorkit"},
    "CircuitKIT": {"install": "circuitkit",      "import": "circuitkit"},
    "SafeTune":   {"install": "safetune",        "import": "safetune"},
    "AuditKIT":   {"install": "auditkit",        "import": "auditkit"},
}
ENV_BEFORE = sp.env_info()

python 3.13.16


In [4]:
import importlib, inspect, pkgutil, subprocess

report = {}
for name, spec in TOOLKITS.items():
    entry = {"install": spec["install"], "import": spec["import"]}
    if not spec["install"] or not spec["import"]:
        entry["status"] = "not configured"
        report[name] = entry
        continue
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec["install"].split()],
                       capture_output=True, text=True)
    entry["pip_ok"] = r.returncode == 0
    if r.returncode != 0:
        entry["pip_error"] = r.stderr[-1500:]
    dist = spec["install"].split("[")[0]
    show = subprocess.run([sys.executable, "-m", "pip", "show", dist], capture_output=True, text=True).stdout
    entry["pip_show"] = {k.strip(): v.strip() for k, _, v in (l.partition(":") for l in show.splitlines())
                         if k.strip() in ("Name", "Version", "Home-page", "Author", "License", "Requires")}
    try:
        mod = importlib.import_module(spec["import"])
        entry["version"] = getattr(mod, "__version__", None)
        entry["public"] = sorted(n for n in dir(mod) if not n.startswith("_"))
        if hasattr(mod, "__path__"):
            entry["submodules"] = sorted(m.name for m in pkgutil.walk_packages(mod.__path__, mod.__name__ + "."))[:200]
        entry["status"] = "imported"
    except Exception as e:
        entry["status"] = f"import failed: {type(e).__name__}: {e}"
    report[name] = entry
    print(f"{name}: {entry['status']}")

env_after = sp.env_info()
changed = {k: (ENV_BEFORE.get(k), v) for k, v in env_after.items() if ENV_BEFORE.get(k) != v}
print("env changed by installs:", changed or "nothing")
sp.save_json({"toolkits": report, "python": platform.python_version(), "env_before": ENV_BEFORE,
              "env": env_after}, sp.out_dir("toolkits") / "toolkit_apis.json")

CuratorKIT: imported


CircuitKIT: imported


I0000 00:00:1793981048.809306      19 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


SafeTune: imported

AuditKIT: imported

env changed by installs:

nothing

PosixPath('/kaggle/working/toolkits/toolkit_apis.json')

In [5]:
# Inspect one object in detail: set TARGET to e.g. "circuitkit.discover" once you know the module names.
TARGET = ""
if TARGET:
    mod_name, _, attr = TARGET.rpartition(".")
    obj = getattr(importlib.import_module(mod_name), attr)
    print(inspect.signature(obj) if callable(obj) else obj)
    print(inspect.getdoc(obj))

## Questions to answer (record the answers in docs/03 §5)

- **CircuitKIT:** Can it take our own clean/corrupt prompt pairs and our own metric (refusal logit difference)?
  Which models does it support? What does a discovered circuit look like (heads, MLPs, edges)?
- **SafeTune:** Which weight-space repair methods exist (patching, merging, projection)? Can a repair be
  restricted to chosen components? Does it have its own safety-neuron/circuit locator we can cross-check with?
- **AuditKIT:** Does it bundle a harmfulness judge, HarmBench/AdvBench-style sets and an over-refusal set
  (XSTest)? Can it score a model in memory or only from a Hub id / path?
- **CuratorKIT:** Can it generate/clean instruction data from a public corpus and dedupe? Output format?

Until these are answered, notebooks 05–10 use the plain-PyTorch fallbacks in sp_utils (docs/03 §5 *Fallbacks*).
Each notebook has a marked cell where the toolkit call goes once its API is known.